<a href="https://colab.research.google.com/github/mesata/Facial-Expression-Recognition-/blob/main/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from google.colab import drive
import wandb

drive.mount('/content/drive')
save_path = '/content/drive/MyDrive/fer2013_processed'

X_train = np.load(f'{save_path}/X_train.npy')
y_train = np.load(f'{save_path}/y_train.npy')
X_val = np.load(f'{save_path}/X_val.npy')
y_val = np.load(f'{save_path}/y_val.npy')


Mounted at /content/drive


In [2]:

class_counts = np.bincount(y_train)
class_weights = len(y_train) / (len(class_counts) * class_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

X_train_pt = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
y_train_pt = torch.tensor(y_train, dtype=torch.long)
X_val_pt = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)
y_val_pt = torch.tensor(y_val, dtype=torch.long)

BATCH_SIZE = 64
train_loader = DataLoader(TensorDataset(X_train_pt, y_train_pt), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_pt, y_val_pt), batch_size=BATCH_SIZE, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" ყველაფერი მზადაა. სამუშაო გარემო: {device}")

 ყველაფერი მზადაა. სამუშაო გარემო: cuda


In [3]:
def train_model(model, model_name, lr=0.003, epochs=5):
    run = wandb.init(
        project="facial-expression-recognition",
        name=model_name,
        config={"learning_rate": lr, "epochs": epochs, "batch_size": 64}
    )

    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
    optimizer = optim.Adam(model.parameters(), lr=lr)

    print(f"წვრთნა დაიწყო")

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, correct_train, total_train = 0.0, 0, 0

        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            if epoch == 1 and batch_idx == 0:
                print(f" [Forward Check] შემავალი ზომა: {images.shape}")
                outputs = model(images)
                print(f" [Forward Check] გამომავალი ზომა: {outputs.shape}")
            else:
                outputs = model(images)

            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total_train += labels.size(0)
            correct_train += predicted.eq(labels).sum().item()

        model.eval()
        val_loss, correct_val, total_val = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                total_val += labels.size(0)
                correct_val += predicted.eq(labels).sum().item()

        t_loss = train_loss / total_train
        t_acc = correct_train / total_train
        v_loss = val_loss / total_val
        v_acc = correct_val / total_val

        print(f"Epoch {epoch}/{epochs} -> Train Loss: {t_loss:.4f}, Train Acc: {t_acc:.4f} | Val Loss: {v_loss:.4f}, Val Acc: {v_acc:.4f}")

        wandb.log({
            "epoch": epoch,
            "train_loss": t_loss,
            "train_accuracy": t_acc,
            "val_loss": v_loss,
            "val_accuracy": v_acc
        })

    run.finish()
    print(f" {model_name} მორჩა და დაილოგა \n")

In [4]:
class Architecture1(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Linear(16 * 24 * 24, 7)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

model1 = Architecture1().to(device)
train_model(model1, model_name="Arch1_Tiny_Baseline", lr=0.003, epochs=5)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mesatia (mesatia-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


წვრთნა დაიწყო
 [Forward Check] შემავალი ზომა: torch.Size([64, 1, 48, 48])
 [Forward Check] გამომავალი ზომა: torch.Size([64, 7])
Epoch 1/5 -> Train Loss: 1.8264, Train Acc: 0.2977 | Val Loss: 1.6915, Val Acc: 0.3529
Epoch 2/5 -> Train Loss: 1.6130, Train Acc: 0.3946 | Val Loss: 1.5977, Val Acc: 0.3984
Epoch 3/5 -> Train Loss: 1.4798, Train Acc: 0.4475 | Val Loss: 1.5578, Val Acc: 0.3949
Epoch 4/5 -> Train Loss: 1.3708, Train Acc: 0.4833 | Val Loss: 1.5900, Val Acc: 0.4305
Epoch 5/5 -> Train Loss: 1.2990, Train Acc: 0.5088 | Val Loss: 1.5513, Val Acc: 0.4411


epoch,▁▃▅▆█
train_accuracy,▁▄▆▇█
train_loss,█▅▃▂▁
val_accuracy,▁▅▄▇█
val_loss,█▃▁▃▁
epoch,5
train_accuracy,0.50881
train_loss,1.29904
val_accuracy,0.44114
val_loss,1.55133


 Arch1_Tiny_Baseline მორჩა და დაილოგა 



epochebis matebastan ertad train acc sakmaod izrdeba, val acc mastan shedarebit bevrad mcired. Overfittings aketebs. magram tan Train acc-ic da val acc-ic dzalian dabalia. models gartuleba chirdeba. davamateb fenas

In [5]:
class Architecture2(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(64 * 12 * 12, 128),
            nn.ReLU(),
            nn.Linear(128, 7)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

model2 = Architecture2().to(device)
train_model(model2, model_name="Arch2_deeper_CNN", lr=0.001, epochs=5)


წვრთნა დაიწყო
 [Forward Check] შემავალი ზომა: torch.Size([64, 1, 48, 48])
 [Forward Check] გამომავალი ზომა: torch.Size([64, 7])
Epoch 1/5 -> Train Loss: 1.8349, Train Acc: 0.2725 | Val Loss: 1.7593, Val Acc: 0.3209
Epoch 2/5 -> Train Loss: 1.6869, Train Acc: 0.3656 | Val Loss: 1.6286, Val Acc: 0.3836
Epoch 3/5 -> Train Loss: 1.5767, Train Acc: 0.4043 | Val Loss: 1.5679, Val Acc: 0.4233
Epoch 4/5 -> Train Loss: 1.4730, Train Acc: 0.4424 | Val Loss: 1.5373, Val Acc: 0.4511
Epoch 5/5 -> Train Loss: 1.3977, Train Acc: 0.4652 | Val Loss: 1.4835, Val Acc: 0.4558


epoch,▁▃▅▆█
train_accuracy,▁▄▆▇█
train_loss,█▆▄▂▁
val_accuracy,▁▄▆██
val_loss,█▅▃▂▁
epoch,5
train_accuracy,0.46521
train_loss,1.39773
val_accuracy,0.45577
val_loss,1.48349


 Arch2_deeper_CNN მორჩა და დაილოგა 



uketesia, imxela acdena aghar aqvs train da validations. shedegi isev dabalia

In [6]:
class Architecture3(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 6 * 6, 256),
            nn.ReLU(),
            nn.Linear(256, 7)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

model3 = Architecture3().to(device)
train_model(model3, model_name="Arch3_deeperer_CNN", lr=0.001, epochs=5)

წვრთნა დაიწყო
 [Forward Check] შემავალი ზომა: torch.Size([64, 1, 48, 48])
 [Forward Check] გამომავალი ზომა: torch.Size([64, 7])
Epoch 1/5 -> Train Loss: 1.8229, Train Acc: 0.2796 | Val Loss: 1.7021, Val Acc: 0.3415
Epoch 2/5 -> Train Loss: 1.6332, Train Acc: 0.3852 | Val Loss: 1.5687, Val Acc: 0.3836
Epoch 3/5 -> Train Loss: 1.5058, Train Acc: 0.4324 | Val Loss: 1.5042, Val Acc: 0.4504
Epoch 4/5 -> Train Loss: 1.4008, Train Acc: 0.4650 | Val Loss: 1.4911, Val Acc: 0.4706
Epoch 5/5 -> Train Loss: 1.3011, Train Acc: 0.4982 | Val Loss: 1.4062, Val Acc: 0.4767


epoch,▁▃▅▆█
train_accuracy,▁▄▆▇█
train_loss,█▅▄▂▁
val_accuracy,▁▃▇██
val_loss,█▅▃▃▁
epoch,5
train_accuracy,0.4982
train_loss,1.30112
val_accuracy,0.47667
val_loss,1.40624


 Arch3_deeperer_CNN მორჩა და დაილოგა 



gauareseba daiwyo mexute epochaze. jer axali arqiteqturis cda ar minda, amis hiperparametrs shevcvli. + dzaan nela midis da gpuze gadasvlaa sawiro.


In [7]:
model3_tuned = Architecture3().to(device)
train_model(model3_tuned, model_name="Arch3_Tuned_LowLR", lr=0.0005, epochs=7)

წვრთნა დაიწყო
 [Forward Check] შემავალი ზომა: torch.Size([64, 1, 48, 48])
 [Forward Check] გამომავალი ზომა: torch.Size([64, 7])
Epoch 1/7 -> Train Loss: 1.8611, Train Acc: 0.2592 | Val Loss: 1.7442, Val Acc: 0.3453
Epoch 2/7 -> Train Loss: 1.6469, Train Acc: 0.3869 | Val Loss: 1.5964, Val Acc: 0.4130
Epoch 3/7 -> Train Loss: 1.5084, Train Acc: 0.4355 | Val Loss: 1.4843, Val Acc: 0.4509
Epoch 4/7 -> Train Loss: 1.4092, Train Acc: 0.4697 | Val Loss: 1.5542, Val Acc: 0.4901
Epoch 5/7 -> Train Loss: 1.2951, Train Acc: 0.5027 | Val Loss: 1.4388, Val Acc: 0.4479
Epoch 6/7 -> Train Loss: 1.2102, Train Acc: 0.5351 | Val Loss: 1.3370, Val Acc: 0.5147
Epoch 7/7 -> Train Loss: 1.1161, Train Acc: 0.5650 | Val Loss: 1.3937, Val Acc: 0.5196


epoch,▁▂▃▅▆▇█
train_accuracy,▁▄▅▆▇▇█
train_loss,█▆▅▄▃▂▁
val_accuracy,▁▄▅▇▅██
val_loss,█▅▄▅▃▁▂
epoch,7
train_accuracy,0.56499
train_loss,1.11612
val_accuracy,0.51962
val_loss,1.39366


 Arch3_Tuned_LowLR მორჩა და დაილოგა 



jobs winas, magram 6-7 epochebshi ar izrdeba val acc ise, rogorc train acc. overfittings iwyebs. regularizaciebs shemovagdeb

In [8]:
class Architecture4(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.4)
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 6 * 6, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 7)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

model4 = Architecture4().to(device)
train_model(model4, model_name="Arch4_Regularized_v1", lr=0.001, epochs=10)

წვრთნა დაიწყო
 [Forward Check] შემავალი ზომა: torch.Size([64, 1, 48, 48])
 [Forward Check] გამომავალი ზომა: torch.Size([64, 7])
Epoch 1/10 -> Train Loss: 1.8680, Train Acc: 0.2756 | Val Loss: 1.6713, Val Acc: 0.3675
Epoch 2/10 -> Train Loss: 1.6852, Train Acc: 0.3623 | Val Loss: 1.5417, Val Acc: 0.4221
Epoch 3/10 -> Train Loss: 1.5885, Train Acc: 0.3930 | Val Loss: 1.4847, Val Acc: 0.4579
Epoch 4/10 -> Train Loss: 1.5140, Train Acc: 0.4222 | Val Loss: 1.4033, Val Acc: 0.4716
Epoch 5/10 -> Train Loss: 1.4623, Train Acc: 0.4470 | Val Loss: 1.3811, Val Acc: 0.4911
Epoch 6/10 -> Train Loss: 1.4195, Train Acc: 0.4552 | Val Loss: 1.3462, Val Acc: 0.4820
Epoch 7/10 -> Train Loss: 1.3685, Train Acc: 0.4730 | Val Loss: 1.3188, Val Acc: 0.5062
Epoch 8/10 -> Train Loss: 1.3349, Train Acc: 0.4834 | Val Loss: 1.3035, Val Acc: 0.5143
Epoch 9/10 -> Train Loss: 1.3137, Train Acc: 0.4951 | Val Loss: 1.2545, Val Acc: 0.5226
Epoch 10/10 -> Train Loss: 1.2752, Train Acc: 0.5072 | Val Loss: 1.2406, Val Acc

epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▅▅▆▆▇▇██
train_loss,█▆▅▄▃▃▂▂▁▁
val_accuracy,▁▃▅▅▆▆▇▇██
val_loss,█▆▅▄▃▃▂▂▁▁
epoch,10
train_accuracy,0.50717
train_loss,1.27522
val_accuracy,0.53053
val_loss,1.24057


 Arch4_Regularized_v1 მორჩა და დაილოგა 



sauketeso shedegi so far. meti drois micemas vcdi



In [9]:
model4_long = Architecture4().to(device)
train_model(model4_long, model_name="Arch4_Regularized_20Epochs", lr=0.0015, epochs=20)

წვრთნა დაიწყო
 [Forward Check] შემავალი ზომა: torch.Size([64, 1, 48, 48])
 [Forward Check] გამომავალი ზომა: torch.Size([64, 7])
Epoch 1/20 -> Train Loss: 1.8847, Train Acc: 0.2686 | Val Loss: 1.6657, Val Acc: 0.3492
Epoch 2/20 -> Train Loss: 1.7013, Train Acc: 0.3551 | Val Loss: 1.5635, Val Acc: 0.3940
Epoch 3/20 -> Train Loss: 1.5859, Train Acc: 0.3933 | Val Loss: 1.4519, Val Acc: 0.4467
Epoch 4/20 -> Train Loss: 1.5206, Train Acc: 0.4230 | Val Loss: 1.3959, Val Acc: 0.4697
Epoch 5/20 -> Train Loss: 1.4641, Train Acc: 0.4435 | Val Loss: 1.3729, Val Acc: 0.4625
Epoch 6/20 -> Train Loss: 1.4096, Train Acc: 0.4559 | Val Loss: 1.3600, Val Acc: 0.4599
Epoch 7/20 -> Train Loss: 1.3758, Train Acc: 0.4741 | Val Loss: 1.3349, Val Acc: 0.5115
Epoch 8/20 -> Train Loss: 1.3371, Train Acc: 0.4852 | Val Loss: 1.2918, Val Acc: 0.5240
Epoch 9/20 -> Train Loss: 1.3035, Train Acc: 0.4914 | Val Loss: 1.2536, Val Acc: 0.5236
Epoch 10/20 -> Train Loss: 1.2745, Train Acc: 0.5029 | Val Loss: 1.2512, Val Acc

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_accuracy,▁▃▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇███
train_loss,█▇▆▅▅▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁
val_accuracy,▁▂▄▅▄▄▆▆▆▆▇▇▇▇▇▇████
val_loss,█▇▅▄▄▄▃▃▂▂▂▂▂▁▁▁▁▂▁▁
epoch,20
train_accuracy,0.59466
train_loss,1.00078
val_accuracy,0.58858
val_loss,1.17748


 Arch4_Regularized_20Epochs მორჩა და დაილოგა 



kargia magram kide vcdi

In [10]:
BATCH_SIZE = 128
train_loader = DataLoader(TensorDataset(X_train_pt, y_train_pt), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_pt, y_val_pt), batch_size=BATCH_SIZE, shuffle=False)

In [11]:
class Architecture5(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.4)
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 6 * 6, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 7)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

model5 = Architecture5().to(device)
train_model(model5, model_name="Arch5_Max_Capacity", lr=0.0015, epochs=20)

წვრთნა დაიწყო
 [Forward Check] შემავალი ზომა: torch.Size([128, 1, 48, 48])
 [Forward Check] გამომავალი ზომა: torch.Size([128, 7])
Epoch 1/20 -> Train Loss: 1.9051, Train Acc: 0.2499 | Val Loss: 1.6779, Val Acc: 0.3422
Epoch 2/20 -> Train Loss: 1.6866, Train Acc: 0.3481 | Val Loss: 1.5359, Val Acc: 0.4100
Epoch 3/20 -> Train Loss: 1.5845, Train Acc: 0.3894 | Val Loss: 1.4641, Val Acc: 0.4460
Epoch 4/20 -> Train Loss: 1.5081, Train Acc: 0.4185 | Val Loss: 1.4208, Val Acc: 0.4298
Epoch 5/20 -> Train Loss: 1.4622, Train Acc: 0.4361 | Val Loss: 1.3815, Val Acc: 0.4500
Epoch 6/20 -> Train Loss: 1.4123, Train Acc: 0.4506 | Val Loss: 1.3417, Val Acc: 0.5048
Epoch 7/20 -> Train Loss: 1.3599, Train Acc: 0.4662 | Val Loss: 1.3207, Val Acc: 0.4922
Epoch 8/20 -> Train Loss: 1.3230, Train Acc: 0.4797 | Val Loss: 1.2772, Val Acc: 0.5041
Epoch 9/20 -> Train Loss: 1.2809, Train Acc: 0.4913 | Val Loss: 1.2565, Val Acc: 0.5106
Epoch 10/20 -> Train Loss: 1.2471, Train Acc: 0.5050 | Val Loss: 1.2357, Val A

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_accuracy,▁▃▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇███
train_loss,█▆▆▅▅▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁
val_accuracy,▁▃▄▄▄▆▅▆▆▆▇▆▇▇▇█████
val_loss,█▆▅▄▄▃▃▃▂▂▁▁▁▁▁▂▁▁▁▁
epoch,20
train_accuracy,0.60983
train_loss,0.94241
val_accuracy,0.58672
val_loss,1.19346


 Arch5_Max_Capacity მორჩა და დაილოგა 



mexutes meotxem ajoba validaciis setze. meotxes vinaxav saboloo modelad.

In [14]:
BATCH_SIZE = 64
train_loader = DataLoader(TensorDataset(X_train_pt, y_train_pt), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_pt, y_val_pt), batch_size=BATCH_SIZE, shuffle=False)

In [15]:
best_model = Architecture4().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
optimizer = optim.Adam(best_model.parameters(), lr=0.002)

for epoch in range(1, 6):
    best_model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = best_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(best_model.parameters(), max_norm=1.0)
        optimizer.step()

torch.save(best_model.state_dict(), 'best_model.pth')
print(" მეოთხე მოდელის წონები წარმატებით შეინახა სახელით 'best_model.pth'!")

 მეოთხე მოდელის წონები წარმატებით შეინახა სახელით 'best_model.pth'!


In [16]:
run = wandb.init(project="facial-expression-recognition", name="Upload_Arch4_Best_Artifact")
artifact = wandb.Artifact('best_emotion_model', type='model')
artifact.add_file('best_model.pth')
run.log_artifact(artifact)
run.finish()

In [17]:
import time

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
best_model.train()

images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

start_forward = time.time()
outputs = best_model(images)
loss = criterion(outputs, labels)
forward_time = time.time() - start_forward

start_backward = time.time()
loss.backward()
backward_time = time.time() - start_backward

print(f"Forward Pass: {forward_time:.6f} წამი")
print(f"Backward Pass: {backward_time:.6f} წამი")

Forward Pass: 0.010589 წამი
Backward Pass: 0.014725 წამი
